# [실습 5] 인증과 Gated 모델 운영 체크리스트 자동화

## 실습 개요

> "gated 모델을 사용할 때 토큰 권한, 라이선스, 공개 범위를 어떻게 점검할까?"

이번 실습은 Hugging Face 인증과 gated 모델 접근을 운영 관점에서 정리합니다. 실제 토큰 값을 입력하거나 gated 모델을 다운로드하지 않고, 모델 후보 목록과 정책 파일을 읽어 접근 전 확인해야 할 항목을 자동으로 점검합니다.

마지막 실습인 만큼 지금까지 다룬 모델 선택, 배포, 권한 관리, 라이선스 확인을 하나의 체크리스트로 묶습니다. 모델을 잘 불러오는 것만큼, 사용 조건을 지키고 재현 가능한 운영 기록을 남기는 것도 중요합니다.

## 실습 목표

1. read/write/fine-grained 토큰의 사용 목적을 구분한다.
2. gated 모델 접근 전에 필요한 승인과 라이선스 확인 항목을 정리한다.
3. 모델 후보 JSON을 읽고 위험 요소를 자동으로 표시한다.
4. 배포 전 운영 리포트를 Markdown으로 생성한다.
5. [TODO] 코드를 완성해 후보별 접근 가능 여부를 판단한다.

## 데이터 출처

- 모델 후보: `data/model_candidates.json`
- 운영 정책: `config/access_policy.json`

---
## 1. 후보 모델과 접근 정책 읽기

### 토큰을 직접 다루지 않는 안전한 점검 흐름

실습에서는 실제 Hugging Face 토큰 값을 사용하지 않습니다. 대신 토큰이 어떤 scope를 가져야 하는지, gated 모델에서 어떤 승인 조건이 필요한지 정책 파일로 표현합니다.

운영 환경에서는 토큰이 코드나 노트북에 남지 않도록 환경 변수나 secret manager를 사용해야 합니다. 특히 write 권한 토큰은 업로드와 저장소 변경이 가능하므로 최소 권한 원칙에 맞게 관리해야 합니다. 이번 실습의 JSON 정책은 그런 보안 판단을 코드로 자동화하기 위한 작은 형태라고 보면 됩니다.

In [ ]:
import json
from pathlib import Path

candidates = json.loads(Path('data/model_candidates.json').read_text(encoding='utf-8'))
policy = json.loads(Path('config/access_policy.json').read_text(encoding='utf-8'))

print('모델 후보 수:', len(candidates))
print('접근 정책:', policy)

---
## 2. 모델 후보 구조 확인

### gated 여부와 라이선스가 운영 판단에 주는 영향

모델 후보에는 공개 범위, gated 여부, 라이선스, intended use 같은 정보가 들어 있습니다. 성능이 좋아 보여도 라이선스나 접근 조건이 프로젝트 목적과 맞지 않으면 사용할 수 없습니다.

gated 모델은 승인된 계정의 토큰으로만 접근할 수 있고, 승인 조건을 위반하면 접근 권한이 취소될 수 있습니다. 이런 조건은 모델 로딩 코드에서 실패한 뒤에 확인할 문제가 아니라, 모델을 선택하는 단계에서 먼저 걸러야 하는 운영 기준입니다.

In [ ]:
for item in candidates:
    print('모델 ID:', item['model_id'])
    print('  gated 여부:', item['gated'])
    print('  라이선스:', item['license'])
    print('  공개 범위:', item['visibility'])
    print('  사용 목적:', item['intended_use'])

### 후보 모델의 scope와 공개 범위 요약하기

모델 목록이 길어지면 개별 항목을 하나씩 읽는 것만으로는 전체 위험도를 파악하기 어렵습니다. 필요한 token scope와 공개 범위를 집계하면, 어떤 종류의 권한이 많이 요구되는지 빠르게 볼 수 있습니다.

이 요약은 상세 검토 대상을 좁히는 데 유용합니다. 예를 들어 대부분 read scope로 충분한데 일부 후보만 write scope를 요구한다면, 그 후보는 업로드나 저장소 변경이 필요한 작업인지 별도로 확인해야 합니다.

In [ ]:
scope_counts = {}
visibility_counts = {}
for item in candidates:
    scope_counts[item['required_scope']] = scope_counts.get(item['required_scope'], 0) + 1
    visibility_counts[item['visibility']] = visibility_counts.get(item['visibility'], 0) + 1

print('필요 token scope별 후보 수:', scope_counts)
print('공개 범위별 후보 수:', visibility_counts)

---
## 3. [TODO] 접근 가능 여부 판단 함수 만들기

### [TODO] 정책 기반으로 후보를 분류하기

접근 가능 여부는 하나의 조건으로 결정되지 않습니다. gated 모델인지, 승인이 필요한지, 프로젝트에서 허용한 라이선스인지, 현재 작업에 필요한 token scope가 정책과 맞는지를 함께 봐야 합니다.

[TODO]에서는 모델 후보 하나를 받아 `status`와 `reasons`를 반환하는 함수를 완성합니다. 이런 함수를 만들어 두면 새 모델 후보가 추가될 때마다 같은 기준으로 검토할 수 있고, 사람이 매번 체크리스트를 다시 읽으며 판단하는 부담을 줄일 수 있습니다.

판단 흐름은 세 단계입니다. 먼저 gated 모델인데 승인이 없으면 review 대상으로 분류합니다. 다음으로 라이선스가 허용 목록에 있는지 확인합니다. 마지막으로 필요한 token scope가 정책상 허용된 범위인지 검사합니다. 조건을 모두 통과하면 사용할 수 있는 후보로 보고, 하나라도 걸리면 사유를 함께 남깁니다.

완성해야 할 변수는 다음과 같습니다.

- `is_gated`: 후보 모델의 `candidate['gated']` 값입니다.
- `approval_required`: 정책의 `policy['require_approval_for_gated']` 값입니다.
- `is_approved`: 후보 모델의 `candidate['approved']` 값입니다.
- `needs_gated_review`: gated 모델이고, 정책상 승인이 필요하며, 아직 승인되지 않은 경우 `True`입니다.
- `license_name`: 후보 모델의 `candidate['license']` 값입니다.
- `allowed_licenses`: 정책의 `policy['allowed_licenses']` 목록입니다.
- `license_allowed`: `license_name in allowed_licenses` 결과입니다.
- `required_scope`: 후보 모델의 `candidate['required_scope']` 값입니다.
- `allowed_scopes`: 정책의 `policy['allowed_token_scopes']` 목록입니다.
- `scope_allowed`: `required_scope in allowed_scopes` 결과입니다.

In [ ]:
def evaluate_model_access(candidate, policy):
    reasons = []

    # [TODO 1] gated 모델인데 승인이 없으면 reasons에 메시지를 추가하세요.
    is_gated = candidate['gated']
    approval_required = policy['require_approval_for_gated']
    is_approved = candidate['approved']
    needs_gated_review = None
    if needs_gated_review:
        reasons.append('gated approval required')

    # [TODO 2] 라이선스가 허용 목록에 없으면 reasons에 메시지를 추가하세요.
    license_name = candidate['license']
    allowed_licenses = policy['allowed_licenses']
    license_allowed = None
    if not license_allowed:
        reasons.append('license not allowed')

    # [TODO 3] 필요한 토큰 scope가 정책의 허용 scope에 없으면 reasons에 메시지를 추가하세요.
    required_scope = candidate['required_scope']
    allowed_scopes = policy['allowed_scopes']
    scope_allowed = None
    if not scope_allowed:
        reasons.append('token scope not allowed')

    status = 'ok' if not reasons else 'review'
    return {'model_id': candidate['model_id'], 'status': status, 'reasons': reasons}

access_report = [evaluate_model_access(item, policy) for item in candidates]
for row in access_report:
    print('접근 검토 결과:', row)

### review 대상만 따로 모아보기

`review` 상태인 모델은 바로 사용하기보다 승인, 라이선스, 토큰 권한을 추가로 확인해야 합니다. 전체 결과에서 review 대상만 분리하면 후속 조치를 정리하기 쉬워집니다.

실제 운영에서는 이 목록이 승인 요청이나 보안 검토 티켓으로 이어질 수 있습니다. 모델 ID와 사유를 함께 남기는 이유는 담당자가 어떤 조건 때문에 막혔는지 바로 확인할 수 있게 하기 위해서입니다.

In [ ]:
review_items = [row for row in access_report if row['status'] == 'review']
print('추가 검토가 필요한 모델 수:', len(review_items))
for row in review_items:
    print('검토 대상 모델:', row['model_id'], '사유:', ', '.join(row['reasons']))

---
## 4. 운영 리포트 생성

### 모델 사용 조건을 문서로 남기기

모델 접근 검토 결과를 코드 출력으로만 남기면 나중에 추적하기 어렵습니다. Markdown 리포트로 저장하면 리뷰 요청, 승인 기록, 배포 문서에 그대로 연결할 수 있습니다.

여기서는 후보별 상태와 검토 사유를 표로 정리합니다. 실제 조직에서는 데이터 출처, 보안 검토자, 승인 일자, 재검토 주기 같은 항목을 추가하는 경우가 많습니다. 중요한 점은 모델 사용 판단이 개인의 기억이 아니라 기록으로 남아야 한다는 것입니다.

In [ ]:
report_lines = [
    '# Hugging Face 모델 접근 검토 리포트',
    '',
    '| model_id | status | reasons |',
    '|---|---|---|',
]
for row in access_report:
    reasons = ', '.join(row['reasons']) if row['reasons'] else 'none'
    report_lines.append(f"| {row['model_id']} | {row['status']} | {reasons} |")

report_path = Path('access_review_report.md')
report_path.write_text('\n'.join(report_lines), encoding='utf-8')
print('생성된 접근 검토 리포트:')
print(report_path.read_text(encoding='utf-8'))

---
## 5. 종합 체크리스트

### 배포 전 마지막 확인 항목

Hub 모델을 활용하는 운영 흐름에서는 기술 검증과 정책 검토가 함께 진행되어야 합니다. 모델이 로드되는지, 추론 품질이 충분한지, 라이선스와 사용 조건이 목적에 맞는지 모두 확인해야 합니다.

아래 체크리스트는 gated 모델뿐 아니라 private 모델, organization 저장소, write 권한 업로드 작업에도 적용할 수 있습니다. 실습에서는 항목을 출력하는 수준이지만, 실제 파이프라인에서는 CI나 배포 승인 단계에 연결해 누락된 조건이 있으면 배포를 멈추게 만들 수 있습니다.

In [ ]:
checklist = [
    '토큰은 환경 변수나 secret manager로 관리한다.',
    '필요 최소 scope의 토큰을 사용한다.',
    'gated 모델은 승인 계정과 사용 조건을 확인한다.',
    '라이선스와 데이터 출처를 모델 카드에 남긴다.',
    'private/public/organization 공개 범위를 배포 목적에 맞게 선택한다.',
]
for i, item in enumerate(checklist, 1):
    print(f'{i}. {item}')

### 체크리스트를 구조화된 데이터로 남기기

문장 목록은 사람이 읽기에는 편하지만 자동화에는 다루기 어렵습니다. 같은 내용을 딕셔너리 목록으로 만들어 두면 이후 CI 검사, 승인 리포트, 대시보드 출력으로 확장하기 쉽습니다.

카테고리를 붙여 두면 보안, 라이선스, 접근 권한, 재현성처럼 서로 다른 담당자가 맡을 항목을 분리할 수 있습니다. 모델 운영 정책이 커질수록 이런 구조화가 중요해집니다. 처음에는 간단한 리스트여도, 나중에는 조직의 승인 워크플로와 연결되는 데이터가 될 수 있습니다.

In [ ]:
structured_checklist = [
    {'category': 'token', 'item': '토큰은 환경 변수나 secret manager로 관리한다.'},
    {'category': 'token', 'item': '필요 최소 scope의 토큰을 사용한다.'},
    {'category': 'gated', 'item': 'gated 모델은 승인 계정과 사용 조건을 확인한다.'},
    {'category': 'license', 'item': '라이선스와 데이터 출처를 모델 카드에 남긴다.'},
    {'category': 'visibility', 'item': 'private/public/organization 공개 범위를 배포 목적에 맞게 선택한다.'},
]

for row in structured_checklist:
    print('체크리스트 항목:', row)